# Feature Engineering Using Scikit-Learn

### Do You Want to Build a Snowman?

Prepare the inputs for a model that predicts snowman height.

<img src="./assets/olaf.jpeg" width="300" alt="Snowman illustration">

This deliberately tiny, fictional dataset is a **preprocessing exercise**, not evidence that lunch or dinner predicts snowman height. Treat all rows as a supplied training sample; do not report a model generalization score from these 14 examples.

**Business connection:** in a housing dataset, the same steps could impute missing floor areas, scale numerical measurements, and encode property types before predicting price. The workflow transfers; the useful features and preprocessing choices depend on the task.

#### Load Packages

In [1]:
import numpy as np
import pandas as pd

#### Load Data

In [2]:
data = {
    "temp": [-3, 5, 0, 7, 3, -1, 1, None, -6, 3, 0, -1, None, -2],
    "lunch": [
        "soup",
        "sandwich",
        "soup",
        "burger",
        "sandwich",
        "soup",
        "cereal",
        "salad",
        "sandwich",
        "burger",
        "soup",
        "cereal",
        "burger",
        "soup",
    ],
    "dinner": [
        "pizza",
        "pizza",
        "noodles",
        None,
        "fishsticks",
        "pizza",
        None,
        "fishsticks",
        "noodles",
        "pizza",
        None,
        "pizza",
        "fishsticks",
        "pizza",
    ],
    "precipitation": [
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "no",
    ],
    "height_snowman_cm": [100, 0, 75, 0, 20, 25, 0, 35, 170, 0, 85, 85, 45, 0],
}

df_train = pd.DataFrame(data=data).fillna(np.nan)
df_train

,temp,lunch,dinner,precipitation,height_snowman_cm
0,-3.0,soup,pizza,yes,100
1,5.0,sandwich,pizza,no,0
2,0.0,soup,noodles,yes,75
3,7.0,burger,NaN,yes,0
4,3.0,sandwich,fishsticks,yes,20
5,-1.0,soup,pizza,yes,25
6,1.0,cereal,NaN,no,0
7,NaN,salad,fishsticks,yes,35
8,-6.0,sandwich,noodles,yes,170
9,3.0,burger,pizza,no,0


## Exercise

1. Separate `df_train` into `X_train` and `y_train`; the target is `height_snowman_cm`.
2. Identify numeric, nominal, and binary inputs, then inspect missing values.
3. Choose and justify an imputation strategy for each incomplete input. Row order has no time meaning here, so interpolation between neighboring rows is not justified.
4. Choose suitable encodings and a scaler. Explain which columns need each transformation and avoid inventing an order for nominal categories.
5. Import the scikit-learn tools you choose, apply them, and combine the transformed columns into a DataFrame.
6. Produce `X_train_fe` with numerical values, no missing values, the same row count and index as `X_train`, and no target column.

Fit on training data only. To transform new rows later, reuse the fitted objects instead of recomputing statistics.

## My Solution

### 1. Separate inputs and target

In [3]:
target = "height_snowman_cm"

# Inputs: every column except the target
X_train = df_train.drop(columns=target)
# Target: the column we want to predict
y_train = df_train[target]

print("X_train:", X_train.shape, "| y_train:", y_train.shape)

X_train: (14, 4) | y_train: (14,)


### 2. Column types and missing values

In [4]:
# Data type of each input column
X_train.dtypes

temp             float64
lunch                str
dinner               str
precipitation        str
dtype: object

In [5]:
# How many values are missing per column?
X_train.isna().sum()

temp             2
lunch            0
dinner           3
precipitation    0
dtype: int64

In [6]:
# Which categories occur? (dropna=False also counts the missing ones)
for column in ["lunch", "dinner", "precipitation"]:
    print(X_train[column].value_counts(dropna=False).to_dict())

{'soup': 5, 'sandwich': 3, 'burger': 3, 'cereal': 2, 'salad': 1}
{'pizza': 6, nan: 3, 'fishsticks': 3, 'noodles': 2}
{'yes': 10, 'no': 4}


| Column | Type | Missing | Plan |
|---|---|---|---|
| `temp` | numeric (degrees) | 2 of 14 | median imputation, then standard scaling |
| `lunch` | nominal (5 meals, no order) | 0 | one-hot encoding |
| `dinner` | nominal (3 meals, no order) | 3 of 14 | explicit `"unknown"` category, then one-hot encoding |
| `precipitation` | binary (yes / no) | 0 | one 0/1 column (`drop="if_binary"`) |

pandas 3 shows the text columns as `str` (older material says `object`).

### 3. Imputation choices

- **`temp`: training median.** The median is not pulled by single cold or warm days. Row order has no time meaning, so interpolating between neighbouring rows would be guessing.
- **`dinner`: a constant `"unknown"` category.** With only 14 rows, the mode (`pizza`) would add three invented pizzas (pizza would go from 6 to 9 rows). A separate category keeps the fact that dinner was not recorded, and the model can learn whether that matters. The mode would also be a valid, simpler choice.

### 4. Encoding and scaling choices

- **One-hot encoding** for `lunch` and `dinner`: meals have no order, so each meal gets its own 0/1 column. Codes like 0, 1, 2 would invent an order and distances.
- We keep **all** meal columns (no `drop="first"`) and set `handle_unknown="ignore"`: an unseen meal then becomes all zeros, which no known meal produces. With `drop="first"` an unseen meal would look exactly like the dropped reference meal.
- **`precipitation`** is binary: `drop="if_binary"` keeps a single `precipitation_yes` column.
- **`StandardScaler`** for `temp`: the values (-6 to 7 degrees) have no extreme outliers, so mean and standard deviation describe them well. Only `temp` is numeric, so only `temp` is scaled.

### 5. Fit each transformer on the training data and apply it

In [7]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [8]:
# temp: learn the median, fill the gaps, then learn mean and std, scale
temp_imputer = SimpleImputer(strategy="median").set_output(transform="pandas")
temp_scaler = StandardScaler().set_output(transform="pandas")

temp_imputed_train = temp_imputer.fit_transform(X_train[["temp"]])
temp_scaled_train = temp_scaler.fit_transform(temp_imputed_train)

print("learned median:", temp_imputer.statistics_)
print("learned mean:", temp_scaler.mean_, "| learned std:", temp_scaler.scale_)

learned median: [0.]
learned mean: [0.42857143] | learned std: [3.17837078]


In [9]:
# dinner: missing -> "unknown", then one 0/1 column per dinner
dinner_imputer = SimpleImputer(
    strategy="constant", fill_value="unknown"
).set_output(transform="pandas")
dinner_encoder = OneHotEncoder(
    handle_unknown="ignore", sparse_output=False
).set_output(transform="pandas")

dinner_imputed_train = dinner_imputer.fit_transform(X_train[["dinner"]])
dinner_encoded_train = dinner_encoder.fit_transform(dinner_imputed_train)
dinner_encoded_train.head()

,dinner_fishsticks,dinner_noodles,dinner_pizza,dinner_unknown
0,0.0,0.0,1.0,0.0
1,0.0,0.0,1.0,0.0
2,0.0,1.0,0.0,0.0
3,0.0,0.0,0.0,1.0
4,1.0,0.0,0.0,0.0


In [10]:
# lunch: one 0/1 column per lunch; an unseen lunch becomes all zeros
lunch_encoder = OneHotEncoder(
    handle_unknown="ignore", sparse_output=False
).set_output(transform="pandas")
lunch_encoded_train = lunch_encoder.fit_transform(X_train[["lunch"]])

# precipitation: yes/no becomes a single 0/1 column
precip_encoder = OneHotEncoder(
    drop="if_binary", sparse_output=False
).set_output(transform="pandas")
precip_encoded_train = precip_encoder.fit_transform(X_train[["precipitation"]])

print(lunch_encoder.get_feature_names_out())
print(precip_encoder.get_feature_names_out())

['lunch_burger' 'lunch_cereal' 'lunch_salad' 'lunch_sandwich' 'lunch_soup']
['precipitation_yes']


### 6. Combine everything into `X_train_fe`

In [11]:
# Glue the transformed pieces side by side; they share the same row index
X_train_fe = pd.concat(
    [temp_scaled_train, lunch_encoded_train, dinner_encoded_train,
     precip_encoded_train],
    axis=1,
)

# Checks from the task: numeric, complete, same rows, no target
assert X_train_fe.index.equals(X_train.index)
assert not X_train_fe.isna().any().any()
assert target not in X_train_fe.columns
assert all(pd.api.types.is_numeric_dtype(t) for t in X_train_fe.dtypes)
print("X_train_fe shape:", X_train_fe.shape)
X_train_fe

X_train_fe shape: (14, 11)


,temp,lunch_burger,lunch_cereal,lunch_salad,lunch_sandwich,lunch_soup,dinner_fishsticks,dinner_noodles,dinner_pizza,dinner_unknown,precipitation_yes
0,-1.078720,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
1,1.438293,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,-0.134840,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
3,2.067546,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
4,0.809040,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
5,-0.449467,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
6,0.179787,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7,-0.134840,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
8,-2.022600,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
9,0.809040,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


Four input columns became **11 numeric columns**: 1 scaled temperature, 5 lunch columns, 4 dinner columns (including `dinner_unknown`) and 1 precipitation column.

### Check Your Transformations on New Inputs

Reuse your fitted objects to transform the two rows below, **without calling `fit` or `fit_transform` again**. Keep the training output columns and the new row indices, and check that no missing values remain. `wrap` is an unseen lunch category: explain whether your encoder raises an error or handles it explicitly. An expected error is a valid observation; do not silently refit on these rows.

In [12]:
# New inputs only: the fitted transformers must not learn from these rows.
X_new = pd.DataFrame(
    {
        "temp": [np.nan, 2.0],
        "lunch": ["soup", "wrap"],
        "dinner": [np.nan, "pizza"],
        "precipitation": ["yes", "no"],
    },
    index=["known", "new_category"],
)
X_new

,temp,lunch,dinner,precipitation
known,NaN,soup,NaN,yes
new_category,2.0,wrap,pizza,no


In [13]:
# Reuse the fitted objects: only transform, never fit, on new rows
def prepare(X):
    temp = temp_scaler.transform(temp_imputer.transform(X[["temp"]]))
    lunch = lunch_encoder.transform(X[["lunch"]])
    dinner = dinner_encoder.transform(dinner_imputer.transform(X[["dinner"]]))
    precip = precip_encoder.transform(X[["precipitation"]])
    return pd.concat([temp, lunch, dinner, precip], axis=1)


X_new_fe = prepare(X_new)

assert X_new_fe.columns.equals(X_train_fe.columns)
assert X_new_fe.index.equals(X_new.index)
assert not X_new_fe.isna().any().any()
X_new_fe

,temp,lunch_burger,lunch_cereal,lunch_salad,lunch_sandwich,lunch_soup,dinner_fishsticks,dinner_noodles,dinner_pizza,dinner_unknown,precipitation_yes
known,-0.134840,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
new_category,0.494413,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


**What happened to the two rows?**

- `known`: the missing `temp` got the **training** median (0.0 degrees), the missing dinner became `dinner_unknown = 1`.
- `new_category`: `wrap` was never seen in training. Because we chose `handle_unknown="ignore"`, all five lunch columns are 0. The encoder encodes it quietly instead of crashing (no error, no warning in our run), and no known lunch looks like this.

For comparison, the **default** encoder (`handle_unknown="error"`) refuses the unseen lunch. We fit it on the training lunches only and catch exactly that expected error:

In [14]:
# Default encoder: an unseen category raises a ValueError
strict_lunch_encoder = OneHotEncoder(sparse_output=False)
strict_lunch_encoder.fit(X_train[["lunch"]])

try:
    strict_lunch_encoder.transform(X_new[["lunch"]])
except ValueError as error:
    print("Expected error:", error)

Expected error: Found unknown categories ['wrap'] in column 0 during transform


### Check Your Understanding

1. Why should you use `transform`, not `fit_transform`, on the test set?
2. Why could encoding meals as 0, 1, 2 mislead a linear or distance-based model?
3. Does `RobustScaler` remove extreme observations? Explain briefly.

Compare your reasoning with the short answers at the end of the solution.

### My Answers

1. **`transform`, not `fit_transform`, on the test set:** `transform` reuses the statistics learned from training data (median, mean, categories). `fit_transform` would relearn them from the test rows, so the test data would shape the preprocessing (data leakage) and the columns could differ from what the model was trained on.
2. **Meals as 0, 1, 2:** the numbers invent an order (soup < sandwich < burger) and equal distances between meals. A linear model would give "burger" twice the effect of "sandwich", and a distance-based model would call soup and sandwich "closer" than soup and burger. One-hot encoding avoids both.
3. **`RobustScaler` does not remove extreme observations.** It subtracts the median and divides by the IQR, two statistics that outliers barely move. The outliers stay in the data, still far away, just on the new scale.